# K03 - Klasifikasi Ordinal Tingkat Kematangan Buah Sawit
**Notebook pipeline lengkap untuk Google Colab** (Visi Komputer IF0PP006, Kelompok 3)

Urutan: siapkan lingkungan → unduh & audit data → baseline → eksperimen CNN → tabel hasil → analisis kesalahan → latensi → demo.

**Cara 1 – browser:** buka di colab.research.google.com → *Runtime → Change runtime type → T4 GPU*.

**Cara 2 – VS Code (ekstensi Google Colab):** buka notebook ini → *Select Kernel → Colab → New Colab Server* → pilih varian **GPU** (T4).
Kode berjalan di **server Google**, bukan di laptop: berkas lokal (folder proyek, `data/`) **tidak terlihat** oleh runtime. Karena itu sel 1b mengambil kode dari GitHub atau dari zip di Google Drive.

Semua sel aman dijalankan ulang; run yang sudah selesai akan dilewati. Simpan `outputs/` ke Google Drive (sel 1a & 1c) agar checkpoint tidak hilang saat runtime diputus.

## 1. Lingkungan

**Bila runtime terputus** (mis. ditinggal semalam): Colab memberi mesin baru dan `/content` kosong. Jalankan ulang **1a** dan **1b**, lalu **2**. Checkpoint di `outputs/` aman di Google Drive, dan run yang sudah selesai otomatis dilewati. Sel 1d boleh dilewati bila tidak mendapat GPU dan pelatihan sudah selesai.


In [ ]:
# 1a. Hubungkan Google Drive (untuk menyimpan outputs/ dan, bila perlu, mengambil zip proyek).
#     VS Code: bila sel ini gagal, buka Command Palette (Ctrl+Shift+P) →
#     "Colab: Mount Google Drive to Server...", lalu jalankan sel berikutnya.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 1b. Ambil kode proyek ke runtime Colab. Pilih SALAH SATU:
#   A) dari GitHub (disarankan): isi REPO_URL.
#   B) dari Google Drive: unggah K03_KematanganSawit.zip ke "My Drive" lalu biarkan REPO_URL kosong.
REPO_URL = ""  # mis. "https://github.com/<akun>/K03_KematanganSawit.git"
ZIP_DI_DRIVE = "/content/drive/MyDrive/K03_KematanganSawit.zip"
import os
%cd /content
if not os.path.exists("K03_KematanganSawit"):
    if REPO_URL:
        !git clone -q {REPO_URL}
    elif os.path.exists(ZIP_DI_DRIVE):
        !unzip -q {ZIP_DI_DRIVE} -d /content
    else:
        raise FileNotFoundError("Isi REPO_URL atau unggah K03_KematanganSawit.zip ke My Drive.")
%cd /content/K03_KematanganSawit
!pip install -q -r requirements.txt

# 1c. Simpan outputs/ di Drive agar checkpoint tidak hilang saat runtime diputus.
!mkdir -p /content/drive/MyDrive/K03_KematanganSawit_outputs
!rm -rf outputs && ln -s /content/drive/MyDrive/K03_KematanganSawit_outputs outputs

In [ ]:
# 1d. WAJIB: pastikan GPU aktif sebelum melatih (CPU ~1 jam per run vs ~10 menit di T4).
import torch, platform
assert torch.cuda.is_available(), (
    "GPU TIDAK aktif! Browser: Runtime → Change runtime type → T4 GPU. "
    "VS Code: Command Palette → 'Colab: Remove Server', lalu Select Kernel → Colab → New Colab Server → GPU.")
print("Python", platform.python_version(), "| torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -m pytest -q   # uji unit + uji integrasi kecil (data sintetis)

## 2. Data: unduh, metadata, audit (CP03, CP07)

In [ ]:
# Runtime Colab yang terputus dimulai dari nol: pastikan proyek sudah disiapkan.
import os
assert os.path.exists("/content/K03_KematanganSawit"), "Runtime baru/terputus: jalankan sel 1a dan 1b dulu (outputs/ aman di Drive)."
%cd /content/K03_KematanganSawit

!python scripts/download_data.py      # ~260 MB dari Mendeley Data, SHA-256 diverifikasi
!python scripts/prepare_data.py
!python scripts/audit_dataset.py      # membuat split blok (split utama) + ringkasan kebocoran

In [ ]:
# Runtime Colab yang terputus dimulai dari nol: pastikan proyek sudah disiapkan.
import os
assert os.path.exists("/content/K03_KematanganSawit"), "Runtime baru/terputus: jalankan sel 1a dan 1b dulu (outputs/ aman di Drive)."
%cd /content/K03_KematanganSawit

from IPython.display import Image, Markdown, display
display(Markdown(open("results/audit/ringkasan_audit.md").read()))
display(Image("results/audit/distribusi_kelas.png"))
display(Image("results/audit/contoh_per_kelas.jpg"))

In [ ]:
# Runtime Colab yang terputus dimulai dari nol: pastikan proyek sudah disiapkan.
import os
assert os.path.exists("/content/K03_KematanganSawit"), "Runtime baru/terputus: jalankan sel 1a dan 1b dulu (outputs/ aman di Drive)."
%cd /content/K03_KematanganSawit

# Periksa manual: pasangan near-duplicate (dHash) dan pasangan bernomor berurutan.
for f in ["contoh_near_duplicate.jpg", "kemiripan_berurutan.png", "contoh_pasangan_berurutan.jpg"]:
    if os.path.exists(f"results/audit/{f}"):
        display(Image(f"results/audit/{f}"))

## 3. Baseline klasik: kelas mayoritas (E0) dan HSV + regresi logistik (E1), pada split blok dan split resmi

In [ ]:
# Runtime Colab yang terputus dimulai dari nol: pastikan proyek sudah disiapkan.
import os
assert os.path.exists("/content/K03_KematanganSawit"), "Runtime baru/terputus: jalankan sel 1a dan 1b dulu (outputs/ aman di Drive)."
%cd /content/K03_KematanganSawit

!python scripts/run_classic_baselines.py                          # split blok (utama)
!python scripts/run_classic_baselines.py --split-scheme official  # split resmi (pembanding)

## 4. Eksperimen CNN (CP08-CP10)
Split utama = **split blok** (audit menemukan 92% gambar test resmi punya tetangga bernomor berurutan di train).

| ID | Konfigurasi | Split | Perbedaan |
|---|---|---|---|
| E2 | `e2_cnn_ce.yaml` | blok | baseline deep learning: cross-entropy |
| E3 | `e3_cnn_corn.yaml` | blok | **metode utama**: loss ordinal CORN (hanya kepala/loss yang berbeda dari E2) |
| E4 | `e4_cnn_corn_hue.yaml` | blok | E3 + augmentasi hue/saturasi |
| E5 | `e5_cnn_corn_balanced.yaml` | blok | E3 + sampler seimbang kelas (iterasi CP11) |
| E6 | `e6_cnn_ce_official.yaml` | resmi | E2 pada split resmi (pembanding kebocoran) |
| E7 | `e7_cnn_corn_official.yaml` | resmi | E3 pada split resmi (pembanding kebocoran) |

Waktu: ~8-12 menit per run di T4; 18 run ≈ 2,5-3,5 jam (bisa dibagi beberapa sesi). Bila waktu terbatas, jalankan dulu `SEEDS = "42"`; urutan `EXPS` = urutan prioritas.

In [ ]:
# Runtime Colab yang terputus dimulai dari nol: pastikan proyek sudah disiapkan.
import os
assert os.path.exists("/content/K03_KematanganSawit"), "Runtime baru/terputus: jalankan sel 1a dan 1b dulu (outputs/ aman di Drive)."
%cd /content/K03_KematanganSawit

SEEDS = "42 43 44"
EXPS = "e2_cnn_ce e3_cnn_corn e4_cnn_corn_hue e5_cnn_corn_balanced e6_cnn_ce_official e7_cnn_corn_official"
for exp in EXPS.split():
    for seed in SEEDS.split():
        if os.path.exists(f"outputs/runs/{exp}/seed{seed}/summary.json"):
            print(f"lewati {exp} seed {seed}")
            continue
        !python -m sawit.train --config configs/{exp}.yaml --seed {seed}

## 5. Tabel hasil beku

In [ ]:
# Runtime Colab yang terputus dimulai dari nol: pastikan proyek sudah disiapkan.
import os
assert os.path.exists("/content/K03_KematanganSawit"), "Runtime baru/terputus: jalankan sel 1a dan 1b dulu (outputs/ aman di Drive)."
%cd /content/K03_KematanganSawit

!python scripts/aggregate_results.py
display(Markdown(open("results/hasil_ringkas.md").read()))

In [ ]:
# Runtime Colab yang terputus dimulai dari nol: pastikan proyek sudah disiapkan.
import os
assert os.path.exists("/content/K03_KematanganSawit"), "Runtime baru/terputus: jalankan sel 1a dan 1b dulu (outputs/ aman di Drive)."
%cd /content/K03_KematanganSawit

display(Image("outputs/runs/e2_cnn_ce/seed42/confusion_test.png"))
display(Image("outputs/runs/e3_cnn_corn/seed42/confusion_test.png"))
display(Image("outputs/runs/e3_cnn_corn/seed42/curves.png"))

## 6. Analisis kesalahan (CP11)

In [ ]:
# Runtime Colab yang terputus dimulai dari nol: pastikan proyek sudah disiapkan.
import os
assert os.path.exists("/content/K03_KematanganSawit"), "Runtime baru/terputus: jalankan sel 1a dan 1b dulu (outputs/ aman di Drive)."
%cd /content/K03_KematanganSawit

for exp in ["e2_cnn_ce", "e3_cnn_corn"]:
    !python scripts/error_analysis.py --run-dir outputs/runs/{exp}/seed42
d = "results/analisis_kesalahan/e3_cnn_corn_seed42"
display(Markdown(open(f"{d}/ringkasan.md").read()))
for f in ["galeri_salah_berat.jpg", "galeri_salah_tetangga.jpg", "galeri_gradcam.jpg"]:
    if os.path.exists(f"{d}/{f}"): display(Image(f"{d}/{f}"))

Isi kolom `kategori_manual` di `results/analisis_kesalahan/e3_cnn_corn_seed42/kategori_manual.csv` setelah melihat galeri.

## 7. Latensi (CP12)

In [ ]:
# Runtime Colab yang terputus dimulai dari nol: pastikan proyek sudah disiapkan.
import os
assert os.path.exists("/content/K03_KematanganSawit"), "Runtime baru/terputus: jalankan sel 1a dan 1b dulu (outputs/ aman di Drive)."
%cd /content/K03_KematanganSawit

!python scripts/benchmark_latency.py --checkpoint outputs/runs/e3_cnn_corn/seed42/best.pt --device cpu
!python scripts/benchmark_latency.py --checkpoint outputs/runs/e3_cnn_corn/seed42/best.pt --device cuda

## 8. Demo (CP12)

In [ ]:
# Runtime Colab yang terputus dimulai dari nol: pastikan proyek sudah disiapkan.
import os
assert os.path.exists("/content/K03_KematanganSawit"), "Runtime baru/terputus: jalankan sel 1a dan 1b dulu (outputs/ aman di Drive)."
%cd /content/K03_KematanganSawit

!python scripts/make_demo_examples.py
# --share membuat tautan publik sementara (72 jam). Hentikan sel untuk mematikan demo.
!python app/app.py --checkpoint outputs/runs/e3_cnn_corn/seed42/best.pt --share

## 9. Simpan artefak
`results/` (kecil, masuk repositori) disalin ke Drive. Checkpoint `outputs/runs/e3_cnn_corn/seed42/best.pt` sudah berada di Drive (`K03_KematanganSawit_outputs/`) untuk demo; jangan di-commit.
Cara ini berfungsi di browser maupun VS Code (`files.download` tidak tersedia di VS Code).

In [ ]:
# Runtime Colab yang terputus dimulai dari nol: pastikan proyek sudah disiapkan.
import os
assert os.path.exists("/content/K03_KematanganSawit"), "Runtime baru/terputus: jalankan sel 1a dan 1b dulu (outputs/ aman di Drive)."
%cd /content/K03_KematanganSawit

!rm -rf /content/drive/MyDrive/K03_KematanganSawit_results && cp -r results /content/drive/MyDrive/K03_KematanganSawit_results
!ls /content/drive/MyDrive/K03_KematanganSawit_results